In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

# =========================
# 1) 读取数据
# =========================
df = pd.read_csv("ad_agg.csv")
df["event_date"] = pd.to_datetime(df["event_date"])

# 统一 month：全部用字符串 "01","02",...，不要 int，也不要 "2025-02"
# 这样和 LightGBM 的 categorical feature 一致
df["month"] = df["event_date"].dt.strftime("%m")

# =========================
# 2) train / test split
# =========================
train = df[df["event_date"] < "2025-01-01"].copy()
test = df[df["event_date"] >= "2025-01-01"].copy()

feature_cols = ["channel", "audience_segment", "month", "spend"]
target = "revenue"

X_train = train[feature_cols].copy()
y_train = train[target].copy()
X_test = test[feature_cols].copy()
y_test = test[target].copy()

# 统一成字符串，避免不同 dtype 混用
for c in ["channel", "audience_segment", "month"]:
    X_train[c] = X_train[c].astype(str)
    X_test[c] = X_test[c].astype(str)

cat_cols = ["channel", "audience_segment", "month"]

# 关键：train + test 一起决定类别集合，避免 mismatch
for c in cat_cols:
    categories = sorted(
        pd.concat([X_train[c], X_test[c]], ignore_index=True).unique().tolist()
    )
    X_train[c] = pd.Categorical(X_train[c], categories=categories)
    X_test[c] = pd.Categorical(X_test[c], categories=categories)

print("X_train dtypes:")
print(X_train.dtypes)
print()
print("X_test dtypes:")
print(X_test.dtypes)

# =========================
# 3) Optuna objective
# =========================
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("lr", 0.01, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "random_state": 42,
        "objective": "regression",
        "metric": "rmse",
        "verbose": -1
    }

    tscv = TimeSeriesSplit(n_splits=5)
    rmse_scores = []

    for tr_idx, vl_idx in tscv.split(X_train):
        X_tr, X_vl = X_train.iloc[tr_idx].copy(), X_train.iloc[vl_idx].copy()
        y_tr, y_vl = y_train.iloc[tr_idx].copy(), y_train.iloc[vl_idx].copy()

        reg = lgb.LGBMRegressor(**params)
        reg.fit(
            X_tr,
            y_tr,
            categorical_feature=cat_cols,
            eval_set=[(X_vl, y_vl)],
            callbacks=[
                lgb.early_stopping(20),
                lgb.log_evaluation(0)
            ]
        )

        pred = reg.predict(X_vl)
        rmse = np.sqrt(mean_squared_error(y_vl, pred))
        rmse_scores.append(rmse)

    return float(np.mean(rmse_scores))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)
best_params = study.best_params
print("Best params:", best_params)

# =========================
# 4) Final model
# =========================
final_params = {
    "random_state": 42,
    "objective": "regression",
    "metric": "rmse",
    **best_params
}

model = lgb.LGBMRegressor(**final_params, verbose=-1)
model.fit(
    X_train,
    y_train,
    categorical_feature=cat_cols,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20),
        lgb.log_evaluation(0)
    ]
)

print("Model trained successfully.")

[I 2026-08-18 14:18:29,490] A new study created in memory with name: no-name-d99253d3-9bce-4ff9-8538-7034cb24ada6
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-pac

X_train dtypes:
channel             category
audience_segment    category
month               category
spend                float64
dtype: object

X_test dtypes:
channel             category
audience_segment    category
month               category
spend                float64
dtype: object
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[5]	valid_0's rmse: 63206.5
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56401.3
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[2]	valid_0's rmse: 58327.1
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[3]	valid_0's rmse: 56504.3
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[7]	valid_0's rmse: 56533.3
Training until validation scores don't improve for 20 rounds
Early stopping, best iterat

/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'e

Early stopping, best iteration is:
[1]	valid_0's rmse: 56266.4
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 58005.2
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[8]	valid_0's rmse: 56406.3
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[12]	valid_0's rmse: 56630.2
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[5]	valid_0's rmse: 63173.6
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56298.1
Training until validation scores don't improve for 20 rounds


/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'e

Early stopping, best iteration is:
[1]	valid_0's rmse: 58093.9
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56257.5
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[7]	valid_0's rmse: 56709.1
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[26]	valid_0's rmse: 63186.9
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56233.9
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 57858.1
Training until validation scores don't improve for 20 rounds


... (31 outputs left out)

Early stopping, best iteration is:
[31]	valid_0's rmse: 56651
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[12]	valid_0's rmse: 63176.5
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56247.1
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 57917.1
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[3]	valid_0's rmse: 56299.6
Training until validation scores don't improve for 20 rounds


/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'e

Early stopping, best iteration is:
[21]	valid_0's rmse: 56518.8
Best params: {'lr': 0.043188030299503286, 'num_leaves': 48, 'max_depth': 8}
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[2]	valid_0's rmse: 57242.7
Model trained successfully.


/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


In [6]:
# 训练后，先保存 category 级别
category_levels = {
    c: list(X_train[c].cat.categories)
    for c in ["channel", "audience_segment", "month"]
}

def predict_revenue(channel, audience_segment, month, spend):
    input_df = pd.DataFrame({
        "channel": [str(channel).strip()],
        "audience_segment": [str(audience_segment).strip()],
        "month": [str(month).strip().zfill(2)],
        "spend": [float(spend)]
    })

    # 关键：预测时也用训练时一致的 category levels
    for c in ["channel", "audience_segment", "month"]:
        input_df[c] = pd.Categorical(
            input_df[c].astype(str),
            categories=category_levels[c]
        )

    # 保证列顺序和训练时一致
    input_df = input_df[["channel", "audience_segment", "month", "spend"]]

    return float(model.predict(input_df)[0])

TOTAL_BUDGET = 10000
SCENARIO_MONTH = "02"

candidate_units = (
    df[["channel", "audience_segment"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

n_units = len(candidate_units)
print("候选组合数量:", n_units)

equal_budget = TOTAL_BUDGET / n_units

scheme_A = candidate_units.copy()
scheme_A["budget"] = equal_budget

scheme_A["predicted_revenue"] = scheme_A.apply(
    lambda row: predict_revenue(
        row["channel"],
        row["audience_segment"],
        SCENARIO_MONTH,
        row["budget"]
    ),
    axis=1
)

scheme_A["predicted_roi"] = scheme_A["predicted_revenue"] / scheme_A["budget"]

print("\n===== Scheme A: Equal Allocation =====")
print(scheme_A)

print("\nTotal Budget:", scheme_A["budget"].sum())
print("Total Predicted Revenue:", scheme_A["predicted_revenue"].sum())

候选组合数量: 25

===== Scheme A: Equal Allocation =====
      channel audience_segment  budget  predicted_revenue  predicted_roi
0   Instagram            35-44   400.0       57591.325603     143.978314
1    LinkedIn              55+   400.0       58859.292681     147.148232
2      Google            25-34   400.0       57024.657645     142.561644
3     YouTube            45-54   400.0       57591.325603     143.978314
4   Instagram            25-34   400.0       57591.325603     143.978314
5     YouTube            18-24   400.0       57591.325603     143.978314
6    LinkedIn            25-34   400.0       58859.292681     147.148232
7     YouTube            35-44   400.0       57591.325603     143.978314
8    Facebook              55+   400.0       58859.292681     147.148232
9     YouTube              55+   400.0       57591.325603     143.978314
10     Google              55+   400.0       57024.657645     142.561644
11  Instagram            45-54   400.0       57591.325603     143.978314


In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

# 重新创建干净数据，避免旧变量残留
df = pd.read_csv("ad_agg.csv")
df["event_date"] = pd.to_datetime(df["event_date"])
df["month"] = df["event_date"].dt.strftime("%m")

train = df[df["event_date"] < "2025-01-01"].copy()
test = df[df["event_date"] >= "2025-01-01"].copy()

feature_cols = ["channel", "audience_segment", "month", "spend"]
target = "revenue"

X_train = train[feature_cols].copy()
y_train = train[target].copy()
X_test = test[feature_cols].copy()
y_test = test[target].copy()

# 全部转成字符串
for c in ["channel", "audience_segment", "month"]:
    X_train[c] = X_train[c].astype(str)
    X_test[c] = X_test[c].astype(str)

cat_cols = ["channel", "audience_segment", "month"]

# 统一 categories
all_categories = {}
for c in cat_cols:
    categories = sorted(
        pd.concat([X_train[c], X_test[c]], ignore_index=True).unique().tolist()
    )
    all_categories[c] = categories

    X_train[c] = pd.Categorical(X_train[c], categories=categories)
    X_test[c] = pd.Categorical(X_test[c], categories=categories)

# 关键断言：必须完全相等
for c in cat_cols:
    print(c, "train categories:", list(X_train[c].cat.categories))
    print(c, "test  categories:", list(X_test[c].cat.categories))
    assert X_train[c].cat.categories.equals(X_test[c].cat.categories)

print(X_train.dtypes)
print(X_test.dtypes)

# 训练时不要再用旧的 X_train / X_test
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("lr", 0.01, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "random_state": 42,
        "objective": "regression",
        "metric": "rmse",
        "verbose": -1
    }

    tscv = TimeSeriesSplit(n_splits=5)
    scores = []

    for tr_idx, vl_idx in tscv.split(X_train):
        X_tr, X_vl = X_train.iloc[tr_idx].copy(), X_train.iloc[vl_idx].copy()
        y_tr, y_vl = y_train.iloc[tr_idx].copy(), y_train.iloc[vl_idx].copy()

        reg = lgb.LGBMRegressor(**params)
        reg.fit(
            X_tr,
            y_tr,
            categorical_feature=cat_cols,
            eval_set=[(X_vl, y_vl)],
            callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)]
        )

        pred = reg.predict(X_vl)
        rmse = np.sqrt(mean_squared_error(y_vl, pred))
        scores.append(rmse)

    return float(np.mean(scores))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)
best_params = study.best_params
print("best params:", best_params)

final_params = {
    "random_state": 42,
    "objective": "regression",
    "metric": "rmse",
    **best_params
}

model = lgb.LGBMRegressor(**final_params, verbose=-1)
model.fit(
    X_train,
    y_train,
    categorical_feature=cat_cols,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(0)]
)

print("model trained successfully")

[I 2026-08-18 14:20:46,981] A new study created in memory with name: no-name-b4a89659-075b-40db-b4ab-19af8f67a95f
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-pac

channel train categories: ['Facebook', 'Google', 'Instagram', 'LinkedIn', 'YouTube']
channel test  categories: ['Facebook', 'Google', 'Instagram', 'LinkedIn', 'YouTube']
audience_segment train categories: ['18-24', '25-34', '35-44', '45-54', '55+']
audience_segment test  categories: ['18-24', '25-34', '35-44', '45-54', '55+']
month train categories: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
month test  categories: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
channel             category
audience_segment    category
month               category
spend                float64
dtype: object
channel             category
audience_segment    category
month               category
spend                float64
dtype: object
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[6]	valid_0's rmse: 63163.9
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is

/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'e

Early stopping, best iteration is:
[1]	valid_0's rmse: 58014.1
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56267.5
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[13]	valid_0's rmse: 56753.6
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[6]	valid_0's rmse: 63139.6
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56312.9
Training until validation scores don't improve for 20 rounds


/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'e

Early stopping, best iteration is:
[1]	valid_0's rmse: 58137
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56242.3
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[7]	valid_0's rmse: 56661.5
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[6]	valid_0's rmse: 63151.8
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56309.6
Training until validation scores don't improve for 20 rounds


... (11 outputs left out)

Early stopping, best iteration is:
[1]	valid_0's rmse: 58159.5
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56201.9
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[15]	valid_0's rmse: 56630
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[6]	valid_0's rmse: 63164.2
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56345.4
Training until validation scores don't improve for 20 rounds


/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'e

Early stopping, best iteration is:
[1]	valid_0's rmse: 58225.3
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 56178
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[7]	valid_0's rmse: 56726.9
best params: {'lr': 0.036894917864058276, 'num_leaves': 66, 'max_depth': 7}
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[2]	valid_0's rmse: 57257.4
model trained successfully


/Users/lana/miniconda3/envs/ds_common/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
